# Build FAISS Index — DINOv2 — Google Colab

**Qué hace este notebook:**
1. Monta Google Drive y descomprime los pósters
2. Carga metadata desde `metadata.csv` (dentro del zip)
3. Encoda los pósters con DINOv2 → vectores de 1024 dims
4. Construye el índice FAISS
5. Guarda `faiss_dinov2.index` e `index_metadata_dinov2.csv` en Drive

**Antes de ejecutar:**
- Activar GPU: `Runtime → Change runtime type → T4 GPU`
- Tener `posters.zip` en `Mi Unidad/Recomendador_Pelis/`

**Nota:** DINOv2 es un modelo de visión puro — solo soporta búsqueda imagen→imagen.

In [ ]:
# Celda 2 — Instalar dependencias
!pip install -q transformers torch torchvision faiss-cpu pandas Pillow tqdm

In [ ]:
# Celda 3 — Montar Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Celda 4 — Descomprimir pósters
import zipfile
from pathlib import Path

ZIP_PATH    = "/content/drive/MyDrive/Recomendador_Pelis/posters.zip"
EXTRACT_DIR = "/content/posters"
POSTERS_DIR = "/content/posters/posters"  # subcarpeta dentro del zip

Path(EXTRACT_DIR).mkdir(exist_ok=True)

print("Descomprimiendo pósters...")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

posters = list(Path(POSTERS_DIR).glob("*.jpg"))
print(f"Pósters disponibles: {len(posters):,}")

In [ ]:
# Celda 5 — Cargar metadata
import pandas as pd
from pathlib import Path

METADATA_PATH = "/content/posters/processed/metadata.csv"

df = pd.read_csv(METADATA_PATH)

# Remap de rutas → apuntar a los pósters descomprimidos en Colab
df["poster_path"] = df["tmdbId"].apply(
    lambda tid: f"{POSTERS_DIR}/{int(tid)}.jpg"
)

# Filtrar solo los que existen en disco
df = df[df["poster_path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)

print(f"Películas con póster disponible: {len(df):,}")
df.head()

In [ ]:
# Celda 6 — Definir Dinov2Encoder y MovieIndex inline
import torch
import numpy as np
import faiss
from PIL import Image
from transformers import AutoImageProcessor, AutoModel
from tqdm import tqdm

print(f"GPU disponible: {torch.cuda.is_available()}")

MODEL_ID = "facebook/dinov2-large"


class Dinov2Encoder:
    def __init__(self, model_id=MODEL_ID, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Cargando DINOv2 en {self.device}...")
        self.model = AutoModel.from_pretrained(model_id).to(self.device)
        self.processor = AutoImageProcessor.from_pretrained(model_id)
        self.model.eval()

    def _image_features(self, pixel_values):
        outputs = self.model(pixel_values=pixel_values)
        # DINOv2: usar el CLS token como representación global de la imagen
        feats = outputs.last_hidden_state[:, 0, :]
        return feats / feats.norm(dim=-1, keepdim=True)

    @torch.no_grad()
    def encode_images_batch(self, image_paths, batch_size=32):
        all_features = []
        for i in tqdm(range(0, len(image_paths), batch_size), desc="Codificando pósters"):
            batch = image_paths[i : i + batch_size]
            images = [Image.open(p).convert("RGB") for p in batch]
            inputs = self.processor(images=images, return_tensors="pt").to(self.device)
            feats = self._image_features(inputs["pixel_values"])
            all_features.append(feats.cpu().numpy())
        return np.vstack(all_features)


class MovieIndex:
    def __init__(self, dim=1024):
        self.index = faiss.IndexFlatIP(dim)
        self.metadata = None

    def build(self, embeddings, metadata):
        self.index.add(embeddings.astype(np.float32))
        self.metadata = metadata.reset_index(drop=True)

    def save(self, index_path, metadata_path):
        faiss.write_index(self.index, index_path)
        self.metadata.to_csv(metadata_path, index=False)
        print(f"Índice guardado:   {index_path}")
        print(f"Metadata guardada: {metadata_path}")

In [ ]:
# Celda 7 — Encodear con DINOv2
encoder    = Dinov2Encoder()
embeddings = encoder.encode_images_batch(df["poster_path"].tolist())

print(f"Shape embeddings: {embeddings.shape}")  # esperado: (N, 1024)

In [ ]:
# Celda 8 — Construir y guardar el índice
INDEX_PATH = "/content/drive/MyDrive/Recomendador_Pelis/faiss_dinov2.index"
META_PATH  = "/content/drive/MyDrive/Recomendador_Pelis/index_metadata_dinov2.csv"

idx = MovieIndex(dim=embeddings.shape[1])
idx.build(embeddings, df)
idx.save(INDEX_PATH, META_PATH)

print(f"\nListo. {len(df):,} películas indexadas.")

# Backup: descarga directa al navegador
from google.colab import files
files.download(INDEX_PATH)
files.download(META_PATH)

In [ ]:
# Celda 9 — Verificación rápida
sample = df.sample(1, random_state=42).iloc[0]

# Encodear la query
query_img = Image.open(sample["poster_path"]).convert("RGB")
inputs    = encoder.processor(images=query_img, return_tensors="pt").to(encoder.device)
with torch.no_grad():
    query_vec = encoder._image_features(inputs["pixel_values"]).cpu().numpy()

# Buscar vecinos
scores, indices = idx.index.search(query_vec.astype(np.float32), 5)
resultados = df.iloc[indices[0]]

print(f"Query: {sample['title']} ({sample['genres']})")
print()
for i, (_, row) in enumerate(resultados.iterrows()):
    print(f"  {i+1}. {row['title']} ({row['genres']})  — score: {scores[0][i]:.4f}")